## Neural Network Architecture

```text
                    Neural Network

┌─────────────┐
│ Input Layer │
└─────────────┘

 x1      x2      x3      x4
  ●       ●       ●       ●
   \      |      / \      |
    \     |     /   \     |
     \    |    /     \    |
      ▼   ▼   ▼       ▼   ▼

┌───────────────────────────┐
│ Hidden Layer 1 (ReLU)     │
└───────────────────────────┘

      ●      ●      ●      ●
      h1     h2     h3     h4

             │
             ▼

┌───────────────────────────┐
│ Hidden Layer 2 (ReLU)     │
└───────────────────────────┘

         ●      ●      ●
         h1     h2     h3

             │
             ▼

┌───────────────────────────┐
│ Output Layer (Sigmoid)    │
└───────────────────────────┘

                ●
                y

```

### Layer Dimensions

```text
Input Layer      : 4 neurons
Hidden Layer 1   : 4 neurons (ReLU)
Hidden Layer 2   : 3 neurons (ReLU)
Output Layer     : 1 neuron (Sigmoid)

W1 : (4 × 4)
b1 : (1 × 4)

W2 : (3 × 4)
b2 : (1 × 3)

W3 : (1 × 3)
b3 : (1 × 1)
```

### Forward Propagation Flow

```text
X
│
▼
Z1 = XW1ᵀ + b1
│
▼
A1 = ReLU(Z1)
│
▼
Z2 = A1W2ᵀ + b2
│
▼
A2 = ReLU(Z2)
│
▼
Z3 = A2W3ᵀ + b3
│
▼
A3 = Sigmoid(Z3)
│
▼
Prediction
```

## Backpropagation Flow

```text
                    BACKPROPAGATION

                           Loss
                             ↑
                             │
                      dZ3 = A3 - y
                             │
                             ↑
                    ┌────────┴────────┐
                    │ Output Layer    │
                    │ (Sigmoid)       │
                    └────────┬────────┘
                             │
                 dW3 = dZ3ᵀ·A2 / m
                 db3 = ΣdZ3 / m
                             │
                             ↑
                    dA2 = dZ3·W3
                             │
                             ↑
              dZ2 = dA2 ⊙ ReLU'(Z2)
                             │
                             ↑
                   ┌─────────┴─────────┐
                   │ Hidden Layer 2    │
                   │ (3 Neurons)       │
                   └─────────┬─────────┘
                             │
                 dW2 = dZ2ᵀ·A1 / m
                 db2 = ΣdZ2 / m
                             │
                             ↑
                    dA1 = dZ2·W2
                             │
                             ↑
              dZ1 = dA1 ⊙ ReLU'(Z1)
                             │
                             ↑
                   ┌─────────┴─────────┐
                   │ Hidden Layer 1    │
                   │ (4 Neurons)       │
                   └─────────┬─────────┘
                             │
                  dW1 = dZ1ᵀ·X / m
                  db1 = ΣdZ1 / m
                             │
                             ↑
                           Input
```

### Gradient Descent Updates

```text
W3 = W3 - lr·dW3
b3 = b3 - lr·db3

W2 = W2 - lr·dW2
b2 = b2 - lr·db2

W1 = W1 - lr·dW1
b1 = b1 - lr·db1
```

### Error Flow Summary

```text
Loss
 ↑
dZ3
 ↑
dA2
 ↑
dZ2
 ↑
dA1
 ↑
dZ1
```

### Parameter Gradient Summary

```text
dW3 ← dZ3 and A2
db3 ← dZ3

dW2 ← dZ2 and A1
db2 ← dZ2

dW1 ← dZ1 and X
db1 ← dZ1
```

In [1]:
import numpy as np

X = np.array([
    [0, 0, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0],
    [0, 0, 1, 1],
    [0, 1, 0, 0],
    [0, 1, 0, 1],
    [0, 1, 1, 0],
    [0, 1, 1, 1],
    [1, 0, 0, 0],
    [1, 0, 0, 1],
    [1, 0, 1, 0],
    [1, 0, 1, 1],
    [1, 1, 0, 0],
    [1, 1, 0, 1],
    [1, 1, 1, 0],
    [1, 1, 1, 1]
])

# Output = 1 if sum of inputs >= 3
y = np.array([
    [0],
    [0],
    [0],
    [0],
    [0],
    [0],
    [0],
    [1],
    [0],
    [0],
    [0],
    [1],
    [0],
    [1],
    [1],
    [1]
])

In [2]:
input_layer = 4
output_layer = 1
hidden1 = 4
hidden2 = 3
epochs = 10000
lr = 0.1
w1 = np.random.randn(input_layer,hidden1)
b1 = np.zeros((1,hidden1))
w2 = np.random.randn(hidden1,hidden2)
b2 = np.zeros((1,hidden2))
w3 = np.random.randn(hidden2,output_layer)
b3 = np.zeros((1,1))


In [6]:
for i in range(epochs):
    z1 = np.dot(X, w1) + b1
    a1 = np.maximum(0, z1)
    z2 = np.dot(a1, w2) + b2
    a2 = np.maximum(0, z2)
    z3 = np.dot(a2, w3) + b3
    a3 = 1 / (1 + np.exp(-z3))

    # Loss

    loss = -np.sum(
        y * np.log(a3 + 1e-8) +
        (1-y) * np.log(1-a3 + 1e-8)
    ) / X.shape[0]

    dw3 = np.dot(a2.T, (a3 - y)) / X.shape[0]
    db3 = np.sum(a3 - y) / X.shape[0]

    da2 = np.dot((a3 - y), w3.T)
    dz2 = da2 * (z2 > 0).astype(float)
    dw2 = np.dot(a1.T, dz2) / X.shape[0]
    db2 = np.sum(dz2, axis=0, keepdims=True) / X.shape[0]

    da1 = np.dot(dz2, w2.T)
    dz1 = da1 * (z1 > 0).astype(float)
    dw1 = np.dot(X.T, dz1) / X.shape[0]
    db1 = np.sum(dz1, axis=0, keepdims=True) / X.shape[0]

    # Gradient Descent

    w3 -= lr * dw3
    b3 -= lr * db3

    w2 -= lr * dw2
    b2 -= lr * db2

    w1 -= lr * dw1
    b1 -= lr * db1

In [7]:
print('w1',w1)
print('w2',w2)
print('w3',w3)
print('b1',b1)
print('b2',b2)
print('b3',b3)


w1 [[-2.18985838 -0.04367914 -0.88140754  0.55693067]
 [ 1.90985332 -2.37850224 -0.98610123  1.19925062]
 [-0.93388689 -1.93308259  0.52515616  1.66191327]
 [ 0.42275815  1.79312099 -1.58404021 -0.82644241]]
w2 [[ 0.7263208   0.72801242 -0.25988663]
 [-0.53996191  1.94597939 -0.66459939]
 [ 2.38127829  1.03860162  1.68557471]
 [-0.83137139 -0.78519705 -0.41108408]]
w3 [[-2.60110356]
 [-2.21264953]
 [-1.28533833]]
b1 [[-0.07923337  0.36155599  2.04134274 -0.27457903]]
b2 [[0.47965851 0.40163258 0.1060314 ]]
b3 [[1.60836676]]
